In [14]:
import pandas as pd
import numpy as np
from statsmodels.genmod.generalized_estimating_equations import GEE
from statsmodels.genmod.families import Binomial
from statsmodels.genmod.cov_struct import Exchangeable
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_recall_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [15]:
acc = pd.read_csv('/Users/sheelareddy/cognitiveCrash/data_cog/acc data update version/acc_data.csv')
sleep = pd.read_csv('/Users/sheelareddy/cognitiveCrash/data_cog/acc data update version/sleep_data.csv')
healthy = pd.read_csv('/Users/sheelareddy/cognitiveCrash/data_cog/acc data update version/healthy aging EMA study dataset.csv')

print(healthy['id'].nunique())

102


In [ ]:
healthy['anxious'] = pd.to_numeric(healthy['anxious'], errors='coerce')

healthy['anxious_z'] = healthy.groupby('id')['anxious'].transform(lambda x: (x - x.mean()) / x.std())

healthy['stressed'] = pd.to_numeric(healthy['stressed'], errors='coerce')

healthy['stressed_z'] = healthy.groupby('id')['stressed'].transform(lambda x: (x - x.mean()) / x.std())

healthy['tired'] = pd.to_numeric(healthy['tired'], errors='coerce')

healthy['tired_z'] = healthy.groupby('id')['tired'].transform(lambda x: (x - x.mean()) / x.std())

healthy['depressed'] = pd.to_numeric(healthy['depressed'], errors='coerce')

healthy['depressed_z'] = healthy.groupby('id')['depressed'].transform(lambda x: (x - x.mean()) / x.std())

In [17]:
healthy['mood_composite_z'] = (healthy['anxious_z'] + healthy['stressed_z'] + healthy['tired_z'] + healthy['depressed_z']) / 4

healthy['mindsharp_z'] = healthy.groupby('id')['mindsharp'].transform(lambda x: (x - x.mean()) / x.std())

healthy['go_nogo.dprime'] = pd.to_numeric(healthy['go_nogo.dprime'], errors='coerce')

healthy['gonogo_z'] = healthy.groupby('id')['go_nogo.dprime'].transform(lambda x: (x - x.mean()) / x.std())



In [18]:
#consecutive days in bad mood
#how many consecutive day mood_conmposite_z > 0

healthy = healthy.sort_values(['id', 'day_in_study'])

def count_consecutive_days_mood(group):

    counter = 0
    duration = []

    for mood in group['mood_composite_z']:

        if mood > 0:
            counter += 1
            duration.append(counter)

        else:
            counter = 0
            duration.append(counter)

    return pd.Series(duration, index=group.index)

healthy['days_in_sustained_bad_mood'] = healthy.groupby('id', group_keys=False).apply(count_consecutive_days_mood)


In [19]:
#lagged cognitive featuers

healthy = healthy.sort_values(['id', 'day_in_study']).reset_index(drop=True)

healthy['mindsharp_z_lag1'] = healthy.groupby('id')['mindsharp_z'].shift(1)
healthy['gonogo_z_lag1'] = healthy.groupby('id')['gonogo_z'].shift(1)
healthy['days_in_sustained_bad_mood_lag1'] = healthy.groupby('id')['days_in_sustained_bad_mood'].shift(1)


In [20]:
#DEFINED MODERARTE AND STRICT MOOD CRASHES
healthy['mood_crash_moderate'] = healthy['mood_composite_z'] > 0.75
healthy['mood_crash_strict'] = healthy['mood_composite_z'] > 1

model_data = healthy.dropna(subset=['mindsharp_z', 'gonogo_z', 'mood_composite_z','mood_crash_moderate', 'days_in_sustained_bad_mood','mindsharp_z_lag1', 'gonogo_z_lag1', 'days_in_sustained_bad_mood_lag1'])


print(f"N observations: {len(model_data)}") 
print(f"N participants: {model_data['id'].nunique()}")
print(f"Crash rate: {model_data['mood_crash_moderate'].mean():.1%}")
print(f"Mean days in sustained bad mood: {model_data['days_in_sustained_bad_mood'].mean():.2f}")


model_data['mood_crash_moderate'] = model_data['mood_crash_moderate'].astype(int)
model_data['mood_crash_strict'] = model_data['mood_crash_strict'].astype(int)

model_data['mood_crash_moderate'] = model_data['mood_crash_moderate'].astype(int)
model_data['mood_crash_strict'] = model_data['mood_crash_strict'].astype(int)


N observations: 1056
N participants: 84
Crash rate: 12.0%
Mean days in sustained bad mood: 0.90


In [21]:
#TEST 1 - The below is WRONG it should be the other way.
# Does today's cognition predict today's mood crash?

print("Does today's cognition predict today's mood crash?")

from sklearn.model_selection import GroupKFold
import warnings
warnings.filterwarnings('ignore')

gkf = GroupKFold(n_splits=5)
test1_aucs = []
test1_sens = []
test1_spec = []
test1_prec = []
test1_coefs = []

fold_num = 0

#gkf.split returns a list of train and test rows 
for train_idx, test_idx in gkf.split(model_data, groups=model_data['id']):
    fold_num += 1
    
    train_data = model_data.iloc[train_idx].copy() #copies rows 
    test_data = model_data.iloc[test_idx].copy()
    
    # Fit on training set
    from statsmodels.genmod.generalized_estimating_equations import GEE
    gee_model = GEE.from_formula(
        "mood_crash_moderate ~ mindsharp_z + gonogo_z + days_in_sustained_bad_mood",
        groups="id",
        data=train_data,
        family=Binomial(),
        cov_struct=Exchangeable() #Assumes that any two data points from the same person share a common level of correlation.
    )
   
    result = gee_model.fit()
    
    # Collect coefficients
    test1_coefs.append({
        'mindsharp_z': result.params['mindsharp_z'],
        'gonogo_z': result.params['gonogo_z'],
        'days_in_sustained_bad_mood': result.params['days_in_sustained_bad_mood']
    })

    # Predict on test set
    pred = result.predict(test_data)
    auc = roc_auc_score(test_data['mood_crash_moderate'], pred)
    test1_aucs.append(auc)

    # Confusion matrix: compares predicted values to actual values and outputs a matrix of true positives, false positives, true negatives, and false negatives. Ravel converts the matrix into a 1D array. 
    pred_binary = (pred > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(test_data['mood_crash_moderate'], pred_binary).ravel()
    sens = tp / (tp + fn)
    spec = tn / (tn + fp)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    test1_sens.append(sens)
    test1_spec.append(spec)
    test1_prec.append(prec)
    
    print(f"  Fold {fold_num}: AUC={auc:.3f}, Sens={sens:.3f}, Spec={spec:.3f}, Prec={prec:.3f}")

print(f"\n✓ TEST 1 RESULTS (Average ± Std):")
print(f"  AUC:        {np.mean(test1_aucs):.3f} ± {np.std(test1_aucs):.3f}")
print(f"  Sensitivity: {np.mean(test1_sens):.3f} ± {np.std(test1_sens):.3f}")
print(f"  Specificity: {np.mean(test1_spec):.3f} ± {np.std(test1_spec):.3f}")
print(f"  Precision:   {np.mean(test1_prec):.3f} ± {np.std(test1_prec):.3f}")


coef_df = pd.DataFrame(test1_coefs)

print("TEST 1 AVERAGE COEFFICIENTS:")
print(f"mindsharp_z:                {coef_df['mindsharp_z'].mean():.3f}")
print(f"gonogo_z:                   {coef_df['gonogo_z'].mean():.3f}")
print(f"days_in_sustained_bad_mood: {coef_df['days_in_sustained_bad_mood'].mean():.3f}")
print(f"\nAverage AUC: {np.mean(test1_aucs):.3f}")



Does today's cognition predict today's mood crash?
  Fold 1: AUC=0.853, Sens=0.080, Spec=0.989, Prec=0.500
  Fold 2: AUC=0.787, Sens=0.043, Spec=0.973, Prec=0.167
  Fold 3: AUC=0.741, Sens=0.037, Spec=0.995, Prec=0.500
  Fold 4: AUC=0.847, Sens=0.160, Spec=0.952, Prec=0.308
  Fold 5: AUC=0.756, Sens=0.000, Spec=1.000, Prec=0.000

✓ TEST 1 RESULTS (Average ± Std):
  AUC:        0.797 ± 0.046
  Sensitivity: 0.064 ± 0.054
  Specificity: 0.982 ± 0.017
  Precision:   0.295 ± 0.194
TEST 1 AVERAGE COEFFICIENTS:
mindsharp_z:                -0.721
gonogo_z:                   -0.419
days_in_sustained_bad_mood: 0.461

Average AUC: 0.797


In [22]:
#TEST 2 - WRONG
print("Does today's cognition predict tomorrow's mood crash?")

model_data_lag = model_data.copy()
model_data_lag['mood_crash_moderate_tomorrow'] = model_data_lag.groupby('id')['mood_crash_moderate'].shift(-1)
model_data_lag = model_data_lag.dropna(subset=['mood_crash_moderate_tomorrow'])

print(f"  N observations (with tomorrow available): {len(model_data_lag)}")

test2_aucs = []
test2_sens = []
test2_spec = []
test2_prec = []

fold_num = 0
for train_idx, test_idx in gkf.split(model_data_lag, groups=model_data_lag['id']):
    fold_num += 1
    
    train_data = model_data_lag.iloc[train_idx].copy()
    test_data = model_data_lag.iloc[test_idx].copy()
    
    # Fit on training set
    gee_model = GEE.from_formula(
        "mood_crash_moderate_tomorrow ~ mindsharp_z + gonogo_z + days_in_sustained_bad_mood",
        groups="id",
        data=train_data,
        family=Binomial(),
        cov_struct=Exchangeable()
    )
    result = gee_model.fit()
    
    # Predict on test set
    pred = result.predict(test_data)
    auc = roc_auc_score(test_data['mood_crash_moderate_tomorrow'], pred)
    
    test2_aucs.append(auc)
    
    # Confusion matrix
    pred_binary = (pred > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(test_data['mood_crash_moderate_tomorrow'], pred_binary).ravel()
    sens = tp / (tp + fn)
    spec = tn / (tn + fp)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    test2_sens.append(sens)
    test2_spec.append(spec)
    test2_prec.append(prec)
    
    print(f"  Fold {fold_num}: AUC={auc:.3f}, Sens={sens:.3f}, Spec={spec:.3f}, Prec={prec:.3f}")

print(f"\n✓ TEST 2 RESULTS (Average ± Std):")
print(f"  AUC:        {np.mean(test2_aucs):.3f} ± {np.std(test2_aucs):.3f}")
print(f"  Sensitivity: {np.mean(test2_sens):.3f} ± {np.std(test2_sens):.3f}")
print(f"  Specificity: {np.mean(test2_spec):.3f} ± {np.std(test2_spec):.3f}")
print(f"  Precision:   {np.mean(test2_prec):.3f} ± {np.std(test2_prec):.3f}")


Does today's cognition predict tomorrow's mood crash?
  N observations (with tomorrow available): 972
  Fold 1: AUC=0.538, Sens=0.000, Spec=1.000, Prec=0.000
  Fold 2: AUC=0.538, Sens=0.000, Spec=1.000, Prec=0.000
  Fold 3: AUC=0.432, Sens=0.000, Spec=1.000, Prec=0.000
  Fold 4: AUC=0.563, Sens=0.000, Spec=1.000, Prec=0.000
  Fold 5: AUC=0.515, Sens=0.000, Spec=1.000, Prec=0.000

✓ TEST 2 RESULTS (Average ± Std):
  AUC:        0.517 ± 0.045
  Sensitivity: 0.000 ± 0.000
  Specificity: 1.000 ± 0.000
  Precision:   0.000 ± 0.000


In [23]:
from statsmodels.formula.api import mixedlm
import warnings

warnings.filterwarnings("ignore")

# DATA SETUP

model_setup = model_data.copy()

# Ensure observations are ordered by participant and study day
model_setup = model_setup.sort_values(
    ["id", "day_in_study"]
).reset_index(drop=True)

# Previous available day's cognition
model_setup["mindsharp_z_prior"] = (
    model_setup.groupby("id")["mindsharp_z"]
    .shift(1)
)

# Previous available day's bad mood
model_setup["mood_composite_z_prior"] = (
    model_setup.groupby("id")["mood_composite_z"]
    .shift(1)
)

# Average bad mood across the previous 3 available days
# shift(1) prevents today's mood from entering the rolling average
model_setup["mood_composite_z_prior_avg"] = (
    model_setup.groupby("id")["mood_composite_z"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=3,
            min_periods=3
        ).mean()
    )
)

# ============================================================================
# PERSON-MEAN CENTERING (The Bulletproof Way)
# ============================================================================
# 1. Calculate the true baseline from FULL dataset
person_mean_mood_full = model_setup.groupby('id')['mood_composite_z_prior_avg'].transform('mean')

# 2. Subtract it on the FULL dataset (safe, row-by-row math on complete data)
model_setup['mood_composite_z_prior_avg_centered'] = (
    model_setup['mood_composite_z_prior_avg'] - person_mean_mood_full
)

# 3. NOW drop the missing rows. The centered values are already safely computed!
analysis_data = model_setup.dropna(
    subset=[
        "mindsharp_z",
        "mindsharp_z_prior",
        "mood_composite_z_prior_avg_centered",
        "id"
    ]
).copy()

print("=" * 70)
print("DATA SUMMARY")
print("=" * 70)
print(f"N observations: {len(analysis_data)}")
print(f"N participants: {analysis_data['id'].nunique()}")
print(f"Mean of centered mood: {analysis_data['mood_composite_z_prior_avg_centered'].mean():.4f}")
print("(Baseline calculated from full dataset before dropping missing rows)")

# ============================================================================
# LINEAR MIXED-EFFECTS MODEL (Within-Person Effects)
# ============================================================================

print("\n" + "=" * 70)
print("LINEAR MIXED-EFFECTS MODEL (Within-Person Effects)")
print("Today's Cognition ~ Prior 3-Day Average Bad Mood (person-centered) + Prior Cognition")
print("=" * 70)

# Random intercept for participant
model = mixedlm(
    formula=(
        "mindsharp_z ~ "
        "mood_composite_z_prior_avg_centered + "
        "mindsharp_z_prior"
    ),
    data=analysis_data,
    groups=analysis_data["id"]
)

# ML estimation with a specified optimizer
result = model.fit(
    reml=False,
    method="lbfgs",
    maxiter=2000,
    disp=False
)

print(result.summary())

# ============================================================================
# RESULTS


mood_variable = "mood_composite_z_prior_avg_centered"
cognition_variable = "mindsharp_z_prior"

mood_coef = result.fe_params[mood_variable]
mood_p = result.pvalues[mood_variable]
mood_ci_low, mood_ci_high = result.conf_int().loc[mood_variable]

cognition_coef = result.fe_params[cognition_variable]
cognition_p = result.pvalues[cognition_variable]
cognition_ci_low, cognition_ci_high = result.conf_int().loc[
    cognition_variable
]

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)

print("\nPrior 3-day average bad mood (within-person centered):")
print(f"  Coefficient: {mood_coef:.4f}")
print(f"  p-value: {mood_p:.4f}")
print(
    f"  95% CI: [{mood_ci_low:.4f}, "
    f"{mood_ci_high:.4f}]"
)

if mood_p < 0.05:
    if mood_coef < 0:
        print(
            "  ✓ SIGNIFICANT: When a person is moodier than their own average, "
            "their cognition is significantly LOWER."
        )
    else:
        print(
            "  ✓ SIGNIFICANT: When a person is moodier than their own average, "
            "their cognition is significantly HIGHER (unexpected)."
        )
else:
    print(
        "  ✗ NOT SIGNIFICANT: No within-person effect of bad mood on cognition."
    )

print("\nPrior cognition (persistence):")
print(f"  Coefficient: {cognition_coef:.4f}")
print(f"  p-value: {cognition_p:.4f}")
print(
    f"  95% CI: [{cognition_ci_low:.4f}, "
    f"{cognition_ci_high:.4f}]"
)

if cognition_p < 0.05:
    print("  ✓ SIGNIFICANT: Good cognition on one day predicts good cognition the next.")
else:
    print("  ✗ NOT SIGNIFICANT: No persistence in cognition across days.")

# ============================================================================
# CONVERGENCE CHECK
# ============================================================================

print("\n" + "=" * 70)
print("MODEL STATUS")
print("=" * 70)
print(f"Converged: {result.converged}")

if not result.converged:
    print(
        "WARNING: The model did not converge. "
        "Do not interpret or report its estimates."
    )
else:
    print("✓ The model converged. Estimates may be interpreted.")

DATA SUMMARY
N observations: 805
N participants: 83
Mean of centered mood: -0.0000
(Baseline calculated from full dataset before dropping missing rows)

LINEAR MIXED-EFFECTS MODEL (Within-Person Effects)
Today's Cognition ~ Prior 3-Day Average Bad Mood (person-centered) + Prior Cognition
                   Mixed Linear Model Regression Results
Model:                   MixedLM       Dependent Variable:       mindsharp_z
No. Observations:        805           Method:                   ML         
No. Groups:              83            Scale:                    0.8107     
Min. group size:         2             Log-Likelihood:           -1057.7732 
Max. group size:         15            Converged:                Yes        
Mean group size:         9.7                                                
----------------------------------------------------------------------------
                                    Coef. Std.Err.   z   P>|z| [0.025 0.975]
--------------------------------------

In [24]:
print([
    column for column in healthy.columns
    if "date" in column.lower() or "time" in column.lower()
])

['start_date', 'date_weekend', 'time_of_day']


In [25]:
healthy["date"] = pd.to_datetime(
    healthy["start_date"]
).dt.normalize()

healthy["date"] = pd.to_datetime(healthy["start_date"]).dt.normalize()

healthy.to_csv(
    "/Users/sheelareddy/cognitiveCrash/data_cog/healthy_processed.csv",
    index=False
)



In [26]:
# Make sure date is datetime
healthy["date"] = pd.to_datetime(
    healthy["date"],
    errors="coerce"
).dt.normalize()

# Sort before creating temporal variables
healthy_temporal = (
    healthy
    .sort_values(["id", "date"])
    .reset_index(drop=True)
    .copy()
)

# Average bad mood across previous 3 available health observations
healthy_temporal["prior_3day_bad_mood"] = (
    healthy_temporal
    .groupby("id")["mood_composite_z"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=3,
            min_periods=3
        ).mean()
    )
)

# Cognition from previous available health observation
healthy_temporal["prior_cognition"] = (
    healthy_temporal
    .groupby("id")["mindsharp_z"]
    .shift(1)
)

# Calendar gap from previous available health observation
healthy_temporal["days_since_prior_health"] = (
    healthy_temporal
    .groupby("id")["date"]
    .diff()
    .dt.days
)

In [27]:
healthy_temporal.to_csv(
    "/Users/sheelareddy/cognitiveCrash/data_cog/healthy_processed.csv",
    index=False
)